In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSV files (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Ensure numeric types where appropriate
for df, val_col in [(metric_df, 'value'), (trace_df, 'value'), (log_df, 'value')]:
    if val_col in df.columns:
        df[val_col] = pd.to_numeric(df[val_col], errors='coerce')

# 1) metric.csv aggregation
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False)
metric_counts = metric_grp.agg(count=('value', 'count'),
                               min_timestamp=('timestamp', 'min'),
                               max_timestamp=('timestamp', 'max'))
metric_p95 = metric_grp['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95_value'})
metric_agg = metric_counts.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')
metric_agg_top50 = metric_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']
]

# 2) trace.csv aggregation
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False)
trace_counts = trace_grp.agg(count=('value', 'count'),
                             min_timestamp=('timestamp', 'min'),
                             max_timestamp=('timestamp', 'max'))
trace_p95 = trace_grp['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95_value'})
trace_agg = trace_counts.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
trace_agg_top50 = trace_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'trace_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']
]

# 3) log.csv aggregation
log_grp = log_df.groupby(['cmdb_id', 'log_name'], as_index=False)
log_counts = log_grp.agg(count=('value', 'count'),
                         min_timestamp=('timestamp', 'min'),
                         max_timestamp=('timestamp', 'max'))
log_p95 = log_grp['value'].quantile(0.95).reset_index().rename(columns={'value': 'p95_value'})
log_agg = log_counts.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')
log_agg_top50 = log_agg.sort_values('count', ascending=False).head(50)[
    ['cmdb_id', 'log_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']
]

# 4) error_logs.csv summary and first 20 rows
error_counts = error_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
error_first20 = error_df[['timestamp', 'cmdb_id', 'message']].head(20)

# Return the compact aggregates (variables displayed for IPython)
metric_agg_top50, trace_agg_top50, log_agg_top50, error_counts, error_first20

```
Out[1]:
```
```python
# Build a concise plain-English summary based on the previously computed aggregates
# Reuse existing variables: metric_agg_top50, trace_agg_top50, log_agg_top50, error_counts

def fmt_row(row, cols):
    parts = []
    for c in cols:
        v = row.get(c, None)
        if pd.isna(v):
            parts.append(f"{c}=NaN")
        elif isinstance(v, float):
            parts.append(f"{c}={v:.3g}")
        else:
            parts.append(f"{c}={v}")
    return ", ".join(parts)

# Top 3 examples from each aggregated table (if available)
metric_examples = []
for _, r in metric_agg_top50.head(3).iterrows():
    metric_examples.append(fmt_row(r, ['cmdb_id', 'kpi_name', 'count', 'p95_value']))

trace_examples = []
for _, r in trace_agg_top50.head(3).iterrows():
    trace_examples.append(fmt_row(r, ['cmdb_id', 'trace_name', 'count', 'p95_value']))

log_examples = []
for _, r in log_agg_top50.head(5).iterrows():  # show top 5 logs for clarity
    log_examples.append(fmt_row(r, ['cmdb_id', 'log_name', 'count', 'p95_value']))

if error_counts.shape[0] == 0:
    error_summary = "No error_log records found (error_logs.csv is empty)."
else:
    top_err = error_counts.head(3)
    err_lines = [f"{row.cmdb_id}: count={row['count']}" for _, row in top_err.iterrows()]
    error_summary = "Error log counts (top): " + "; ".join(err_lines)

summary = (
    "Summary of aggregates (compact):\n\n"
    "1) metric.csv:\n"
    f"- Number of metric groups returned: {metric_agg_top50.shape[0]}\n"
    "- Many metric groups have 31 samples (one per minute over the interval).\n"
    "- Top examples: \n  • " + ("\n  • ".join(metric_examples) if metric_examples else "none") + "\n\n"
    "2) trace.csv:\n"
    f"- Number of trace groups returned: {trace_agg_top50.shape[0]}\n"
    "- Most trace groups also have 31 samples.\n"
    "- Top examples: \n  • " + ("\n  • ".join(trace_examples) if trace_examples else "none") + "\n\n"
    "3) log.csv:\n"
    f"- Number of log groups returned: {log_agg_top50.shape[0]}\n"
    "- Common log entries: Tomcat instances show log.row_count and log.error_count (error_count p95 = 0).\n"
    "- Top examples: \n  • " + ("\n  • ".join(log_examples) if log_examples else "none") + "\n\n"
    "4) error_logs.csv:\n"
    f"- {error_summary}\n\n"
    "Notes:\n"
    "- Timestamps are left as integer seconds (UTC+8 was not needed for these aggregates).\n"
    "- P95 values shown are from the pre-filtered full series per group (as requested).\n"
    "- If you want more detail, request specific groups or ask to show additional rows."
)

# Display the plain-English summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                      kpi_name  count  min_timestamp  max_timestamp     p95_value
56       MG01                   OSLinux-CPU_CPU_CPUidleutil     31     1616504400     1616506200  7.362306e+01
21       IG02   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31     1616504400     1616506200  5.855190e+07
20       IG02     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31     1616504400     1616506200  7.944395e+08
36       MG01      JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31     1616504400     1616506200  2.040136e+09
38       MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed     31     1616504400     1616506200  9.245910e+08
..        ...                                           ...    ...            ...            ...           ...
862  Tomcat02    OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31     1616504400     1616506200  3.000000e+00
848  Tomcat02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31     1616504400     1616506200  0.000000e+00
146      MG02      JVM-Threads_7779_JVM_ThreadCount_Threads     31     1616504400     1616506200  7.400000e+01
139      MG01            OSLinux-OSLinux_ZABBIX_Host_Uptime     31     1616504400     1616506200  7.705792e+06
140      MG02      JVM-Memory_7779_JVM_Memory_HeapMemoryMax     31     1616504400     1616506200  2.040136e+09

[50 rows x 6 columns],    cmdb_id                         trace_name  count  min_timestamp  max_timestamp   p95_value
5     IG01               trace.self.row_count     31     1616504400     1616506200  996.000000
4     IG01            trace.self.duration_p95     31     1616504400     1616506200   15.561775
3     IG01           trace.self.duration_mean     31     1616504400     1616506200    4.094347
6     IG01    trace.to_Tomcat01.duration_mean     31     1616504400     1616506200    0.343392
9     IG01    trace.to_Tomcat02.duration_mean     31     1616504400     1616506200    0.340733
..     ...                                ...    ...            ...            ...         ...
94    MG02   trace.from_dockerA2.duration_p95     31     1616504400     1616506200    0.592475
93    MG02  trace.from_dockerA2.duration_mean     31     1616504400     1616506200    0.090009
79    MG02   trace.from_Tomcat01.duration_p95     31     1616504400     1616506200    1.082375
80    MG02      trace.from_Tomcat01.row_count     31     1616504400     1616506200  133.000000
81    MG02  trace.from_Tomcat02.duration_mean     31     1616504400     1616506200    0.235736

[50 rows x 6 columns],      cmdb_id          log_name  count  min_timestamp  max_timestamp  p95_value
0   Tomcat01   log.error_count     31     1616504400     1616506200     0.0000
2   Tomcat01     log.row_count     31     1616504400     1616506200   249.0000
3   Tomcat02   log.error_count     31     1616504400     1616506200     0.0000
5   Tomcat02     log.row_count     31     1616504400     1616506200   258.0000
6   Tomcat03   log.error_count     31     1616504400     1616506200     0.0000
13  apache01     log.row_count     31     1616504400     1616506200   503.0000
14  apache02   log.error_count     31     1616504400     1616506200     0.0000
8   Tomcat03     log.row_count     31     1616504400     1616506200   253.0000
15  apache02     log.row_count     31     1616504400     1616506200   505.0000
12  apache01   log.error_count     31     1616504400     1616506200     0.0000
11  Tomcat04     log.row_count     27     1616504400     1616506200   263.3000
9   Tomcat04   log.error_count     27     1616504400     1616506200     0.0000
1   Tomcat01  log.gc_pause_max      9     1616504520     1616506140     0.0960
10  Tomcat04  log.gc_pause_max      9     1616504520     1616506140     0.0860
7   Tomcat03  log.gc_pause_max      8     1616504520     1616506080     0.0800
4   Tomcat02  log.gc_pause_max      8     1616504520     1616506020     0.0865, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (store it)
metric_global_p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter metric.csv to rows inside window Mar 23, 2021 21:00 to 21:30 (inclusive) in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))

# convert to epoch seconds (UTC)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

metric_window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) For each group compute required stats
# Merge global p95 into windowed rows
metric_window_merged = metric_window_df.merge(metric_global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global_p95 (handle NaN global_p95 => False)
metric_window_merged['is_anomaly'] = (
    (~metric_window_merged['global_p95'].isna()) &
    (metric_window_merged['value'] >= metric_window_merged['global_p95'])
)

# Group to compute requested aggregates
grouped = metric_window_merged.groupby(['cmdb_id', 'kpi_name'], as_index=False)

group_stats = grouped.agg(
    count_in_window=('value', 'count'),
    anomaly_count=('is_anomaly', 'sum'),
    earliest_anomaly_timestamp=('timestamp', lambda s: int(s[metric_window_merged.loc[s.index, 'is_anomaly']].min()) if (metric_window_merged.loc[s.index, 'is_anomaly'].any()) else pd.NA),
    max_value_in_window=('value', 'max'),
    global_p95=('global_p95', 'first')
)

# Calculate max_breach_ratio = (max_value_in_window - global_p95)/global_p95, NaN if global_p95==0 or NaN
group_stats['max_breach_ratio'] = np.where(
    (group_stats['global_p95'].isna()) | (group_stats['global_p95'] == 0),
    np.nan,
    (group_stats['max_value_in_window'] - group_stats['global_p95']) / group_stats['global_p95']
)

# 4) Filter groups that have anomaly_count > 0, sort and select columns
anomalies_df = group_stats[group_stats['anomaly_count'] > 0].copy()

anomalies_df_sorted = anomalies_df.sort_values(
    by=['anomaly_count', 'max_breach_ratio'],
    ascending=[False, False]
)[[
    'cmdb_id', 'kpi_name', 'count_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95', 'max_breach_ratio'
]]

# Limit to top 20 rows for display
anomalies_result = anomalies_df_sorted.head(20)

# 5) Single-line summary
total_groups_scanned = metric_global_p95_df.shape[0]
groups_with_anomalies = anomalies_df.shape[0]
summary = f"Scanned {total_groups_scanned} metric groups; found {groups_with_anomalies} groups with anomalies in the window {start_local} to {end_local} (UTC+8)."

# Display compact results (DataFrame and summary string)
anomalies_result, summary

```
Out[2]:
```
```python
# Prepare a concise plain-English summary using existing kernel variables (anomalies_result, total_groups_scanned, groups_with_anomalies)
try:
    anomalies_result
except NameError:
    import pandas as pd
    anomalies_result = pd.DataFrame()

# Build top example lines (up to 3) from anomalies_result if available
top_lines = []
if not anomalies_result.empty:
    for _, r in anomalies_result.head(3).iterrows():
        top_lines.append(
            f"{r.cmdb_id} | {r.kpi_name} — count_in_window={int(r.count_in_window)}, "
            f"anomaly_count={int(r.anomaly_count)}, earliest_anomaly_ts={int(r.earliest_anomaly_timestamp)}, "
            f"max={r.max_value_in_window:.6g}, p95={r.global_p95:.6g}, breach_ratio={r.max_breach_ratio:.3g}"
        )

summary_text = (
    f"Scanned {total_groups_scanned} metric groups; {groups_with_anomalies} groups showed anomalies in the window "
    f"Mar 23 2021 21:00–21:30 (UTC+8).\n\n"
    f"Key points:\n"
    "- Nearly all flagged groups had anomalies for every sample in the window (many with anomaly_count = 31).\n"
    "- The earliest anomaly timestamps are at the window start (epoch 1616504400).\n"
    "- Highest relative breaches observed (top examples):\n  • " + ("\n  • ".join(top_lines) if top_lines else "none") + "\n\n"
    "- Many JVM and Tomcat metrics were at or equal to their global P95 (breach ratio ≈ 0), while a few OS process metrics (MG01) showed notable relative breaches (e.g., 50% and ~33%).\n\n"
    "Request the detailed anomalies_result DataFrame if you want the full top-20 table."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window    global_p95  max_breach_ratio
129      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31                  1616504400         3.000000e+00  2.000000e+00          0.500000
132      MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31             31                  1616504400         4.000000e-01  3.000000e-01          0.333333
0        IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31                  1616504400         2.040136e+09  2.040136e+09          0.000000
18       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax               31             31                  1616504400         2.040136e+09  2.040136e+09          0.000000
36       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31                  1616504400         2.040136e+09  2.040136e+09          0.000000
39       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31                  1616504400         5.907150e+07  5.907150e+07          0.000000
133      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31                  1616504400         3.000000e+00  3.000000e+00          0.000000
140      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax               31             31                  1616504400         2.040136e+09  2.040136e+09          0.000000
143      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed               31             31                  1616504400         5.837965e+07  5.837965e+07          0.000000
234      MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31                  1616504400         1.000000e+00  1.000000e+00          0.000000
237      MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...               31             31                  1616504400         3.000000e-01  3.000000e-01          0.000000
238      MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31                  1616504400         3.000000e+00  3.000000e+00          0.000000
756  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...               31             31                  1616504400         5.600000e+02  5.600000e+02          0.000000
757  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...               31             31                  1616504400         3.978080e+05  3.978080e+05          0.000000
769  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...               31             31                  1616504400         1.000000e+02  1.000000e+02          0.000000
771  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...               31             31                  1616504400         5.000000e+02  5.000000e+02          0.000000
858  Tomcat02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies               31             31                  1616504400         1.000000e+00  1.000000e+00          0.000000
862  Tomcat02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount               31             31                  1616504400         3.000000e+00  3.000000e+00          0.000000
875  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...               31             31                  1616504400         5.150000e+02  5.150000e+02          0.000000
876  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...               31             31                  1616504400         3.906190e+05  3.906190e+05          0.000000, 'Scanned 1160 metric groups; found 1160 groups with anomalies in the window 2021-03-23 21:00:00+08:00 to 2021-03-23 21:30:00+08:00 (UTC+8).')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric types
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (store in kernel)
trace_global_p95_df = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# 2) Filter trace.csv to the failure window Mar 23, 2021 21:00 to 21:30 (inclusive) in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

trace_window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge global p95 into windowed rows and mark anomalies (value >= global_p95)
trace_window_merged = trace_window_df.merge(trace_global_p95_df, on=['cmdb_id', 'trace_name'], how='left')
trace_window_merged['is_anomaly'] = (~trace_window_merged['global_p95'].isna()) & (trace_window_merged['value'] >= trace_window_merged['global_p95'])

# Compute per-group aggregates
grouped = trace_window_merged.groupby(['cmdb_id', 'trace_name'], as_index=False)

group_stats = grouped.agg(
    count_in_window=('value', 'count'),
    anomaly_count=('is_anomaly', 'sum'),
    max_value_in_window=('value', 'max'),
    global_p95=('global_p95', 'first')
)

# earliest_anomaly_timestamp from anomaly subset
anomaly_subset = trace_window_merged[trace_window_merged['is_anomaly']]
earliest_ts = (
    anomaly_subset
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anomaly_timestamp'})
)

# Merge earliest timestamp (may be missing for groups without anomalies)
group_stats = group_stats.merge(earliest_ts, on=['cmdb_id', 'trace_name'], how='left')

# 3 continued) compute max_breach_ratio
group_stats['max_breach_ratio'] = np.where(
    (group_stats['global_p95'].isna()) | (group_stats['global_p95'] == 0),
    np.nan,
    (group_stats['max_value_in_window'] - group_stats['global_p95']) / group_stats['global_p95']
)

# 4) Filter groups with anomaly_count > 0, sort by max_breach_ratio desc then anomaly_count desc, select top 20
trace_anomalies_df = group_stats[group_stats['anomaly_count'] > 0].copy()

trace_anomalies_result = trace_anomalies_df.sort_values(
    by=['max_breach_ratio', 'anomaly_count'],
    ascending=[False, False]
)[[
    'cmdb_id', 'trace_name', 'count_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95', 'max_breach_ratio'
]].head(20)

# 5) One-line summary
total_trace_groups_scanned = trace_global_p95_df.shape[0]
trace_groups_with_anomalies = trace_anomalies_df.shape[0]
trace_summary = (
    f"Scanned {total_trace_groups_scanned} trace groups; found {trace_groups_with_anomalies} groups with anomalies "
    f"in the window {start_local} to {end_local} (UTC+8)."
)

# Return compact results (limited to top 20) and the summary string
trace_anomalies_result, trace_summary

```
Out[3]:
```
```python
# Build a concise plain-English summary from existing kernel results (trace_anomalies_result, total_trace_groups_scanned, trace_groups_with_anomalies)

try:
    df = trace_anomalies_result
    total = total_trace_groups_scanned
    found = trace_groups_with_anomalies
except NameError:
    df = None
    total = None
    found = None

if df is None or total is None or found is None:
    summary = "Trace anomaly results not found in kernel."
else:
    # Prepare up to top 5 example lines
    examples = []
    for _, r in df.head(5).iterrows():
        examples.append(
            f"{r.cmdb_id} | {r.trace_name} — count_in_window={int(r.count_in_window)}, "
            f"anomaly_count={int(r.anomaly_count)}, earliest_ts={int(r.earliest_anomaly_timestamp)}, "
            f"max={r.max_value_in_window:.6g}, p95={r.global_p95:.6g}, breach_ratio={r.max_breach_ratio:.3f}"
        )
    examples_text = "\n  • ".join(examples) if examples else "none"

    summary = (
        f"Scanned {total} trace groups; {found} groups showed anomalies in the window Mar 23 2021 21:00–21:30 (UTC+8).\n\n"
        "- Observations:\n"
        "  • Many groups had 31 samples in-window and anomaly_count = 2 (i.e., a few points exceeded the global P95).\n"
        "  • The largest relative breaches (top examples):\n"
        f"  • {examples_text}\n\n"
        "- Interpretation: highest breach ratios include ~12.62 (Tomcat01 trace.self.duration_p95) and ~11.42 (Tomcat03 trace.self.duration_p95),\n"
        "  indicating those trace latency P95 metrics exceeded their historical P95 by large multiples during the window.\n"
        "Request the full trace_anomalies_result DataFrame to inspect up to the top-20 groups in detail."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  global_p95  max_breach_ratio
127  Tomcat01            trace.self.duration_p95               31              2                  1616505120             0.526000    0.038625         12.618123
163  Tomcat03            trace.self.duration_p95               31              2                  1616504460             0.422400    0.034000         11.423529
84       MG02  trace.from_Tomcat03.duration_mean               31              2                  1616505120             1.043500    0.203100          4.137850
168  Tomcat03        trace.to_MG02.duration_mean               31              2                  1616505120             1.043500    0.203100          4.137850
181  Tomcat04            trace.self.duration_p95               27              2                  1616505120             0.186750    0.038980          3.790918
36       MG01  trace.from_Tomcat01.duration_mean               31              2                  1616505780             1.063500    0.222907          3.771048
129  Tomcat01        trace.to_MG01.duration_mean               31              2                  1616505780             1.063500    0.222907          3.771048
162  Tomcat03           trace.self.duration_mean               31              2                  1616505900             0.123412    0.038409          2.213059
6        IG01    trace.to_Tomcat01.duration_mean               31              2                  1616504580             1.022000    0.343392          1.976186
120  Tomcat01      trace.from_IG01.duration_mean               31              2                  1616504580             1.022000    0.343392          1.976186
63       MG01           trace.self.duration_mean               31              2                  1616505780             0.303750    0.128717          1.359819
126  Tomcat01           trace.self.duration_mean               31              2                  1616505120             0.101677    0.043163          1.355652
30       IG02    trace.to_Tomcat03.duration_mean               31              2                  1616505720             0.702667    0.317016          1.216503
156  Tomcat03      trace.from_IG02.duration_mean               31              2                  1616505720             0.702667    0.317016          1.216503
72       MG01    trace.to_dockerB1.duration_mean               31              2                  1616505120             0.695000    0.350512          0.982814
219  dockerB1      trace.from_MG01.duration_mean               31              2                  1616505120             0.695000    0.350512          0.982814
46       MG01   trace.from_Tomcat04.duration_p95               27              2                  1616504520             2.082000    1.085995          0.917136
184  Tomcat04         trace.to_MG01.duration_p95               27              2                  1616504520             2.082000    1.085995          0.917136
55       MG01   trace.from_dockerB1.duration_p95               30              2                  1616505600             0.266000    0.144740          0.837778
232  dockerB1         trace.to_MG01.duration_p95               30              2                  1616505600             0.266000    0.144740          0.837778, 'Scanned 255 trace groups; found 255 groups with anomalies in the window 2021-03-23 21:00:00+08:00 to 2021-03-23 21:30:00+08:00 (UTC+8).')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric columns
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute / reuse global P95 per (cmdb_id, trace_name)
try:
    trace_global_p95_df
except NameError:
    trace_global_p95_df = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Compute window timestamps (UTC+8) and filter
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

trace_window_df = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge global p95 and mark anomalies (value >= global_p95)
trace_window_merged = trace_window_df.merge(trace_global_p95_df, on=['cmdb_id', 'trace_name'], how='left')
trace_window_merged['is_anomaly'] = (~trace_window_merged['global_p95'].isna()) & (trace_window_merged['value'] >= trace_window_merged['global_p95'])

# Aggregate per group
grouped = trace_window_merged.groupby(['cmdb_id', 'trace_name'], as_index=False)
group_stats = grouped.agg(
    count_in_window=('value', 'count'),
    anomaly_count=('is_anomaly', 'sum'),
    max_value_in_window=('value', 'max'),
    global_p95=('global_p95', 'first')
)

# earliest_anomaly_timestamp per group (from anomaly subset)
anomaly_subset = trace_window_merged[trace_window_merged['is_anomaly']]
earliest_ts_df = (
    anomaly_subset
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anomaly_timestamp'})
)
group_stats = group_stats.merge(earliest_ts_df, on=['cmdb_id', 'trace_name'], how='left')

# compute max_breach_ratio (NaN if global_p95 is 0 or NaN)
group_stats['max_breach_ratio'] = np.where(
    (group_stats['global_p95'].isna()) | (group_stats['global_p95'] == 0),
    np.nan,
    (group_stats['max_value_in_window'] - group_stats['global_p95']) / group_stats['global_p95']
)

# 4) Filter groups with anomaly_count > 0
trace_anomalies_df = group_stats[group_stats['anomaly_count'] > 0].copy()

# Sort primarily by earliest_anomaly_timestamp asc, secondarily by max_breach_ratio desc
trace_anomalies_ordered = trace_anomalies_df.sort_values(
    by=['earliest_anomaly_timestamp', 'max_breach_ratio'],
    ascending=[True, False]
)[[
    'cmdb_id', 'trace_name', 'count_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'max_value_in_window', 'global_p95', 'max_breach_ratio'
]]

# Top 20 for display
anomalies_ordered = trace_anomalies_ordered.head(20)

# 5) earliest_overall and rows with that earliest timestamp (limit 20)
if trace_anomalies_df.shape[0] > 0:
    earliest_overall = int(trace_anomalies_df['earliest_anomaly_timestamp'].min())
    earliest_rows = trace_anomalies_ordered[trace_anomalies_ordered['earliest_anomaly_timestamp'] == earliest_overall] \
                    .sort_values(by='max_breach_ratio', ascending=False).head(20)
else:
    earliest_overall = pd.NA
    earliest_rows = trace_anomalies_ordered.head(0)

# Return compact outputs (max 20 rows) and earliest_overall
anomalies_ordered, earliest_overall, earliest_rows

```
Out[4]:
```
```python
# Build a concise plain-English summary from the existing anomaly results in the kernel.
import pandas as pd
import pytz

try:
    anomalies_ordered  # sorted anomaly table (top 20 displayed earlier)
    earliest_overall   # earliest overall timestamp (int epoch seconds)
    earliest_rows      # rows that match earliest_overall
    # If trace_anomalies_df exists, use its size; otherwise fall back to anomalies_ordered
    try:
        total_anomaly_groups = int(trace_anomalies_df.shape[0])
    except NameError:
        total_anomaly_groups = int(anomalies_ordered.shape[0])
except NameError:
    summary = "Trace anomaly results are not available in the kernel."
else:
    # Convert earliest_overall to Asia/Shanghai local time string
    if pd.isna(earliest_overall):
        earliest_local_str = "N/A"
    else:
        tz = pytz.timezone('Asia/Shanghai')
        earliest_dt = pd.Timestamp(int(earliest_overall), unit='s', tz=pytz.UTC).astimezone(tz)
        earliest_local_str = earliest_dt.strftime("%Y-03-%d %H:%M:%S %z") if earliest_dt is not None else str(earliest_overall)
        # fix month/day formatting if needed (use actual month/day from timestamp)
        earliest_local_str = earliest_dt.strftime("%Y-%m-%d %H:%M:%S %z")

    # Prepare up to 5 example lines from the ordered anomalies (already sorted by earliest then breach)
    examples = []
    for _, r in anomalies_ordered.head(5).iterrows():
        examples.append(
            f"{r.cmdb_id} | {r.trace_name} — count_in_window={int(r.count_in_window)}, "
            f"anomaly_count={int(r.anomaly_count)}, earliest_ts={int(r.earliest_anomaly_timestamp)}, "
            f"max={r.max_value_in_window:.6g}, p95={r.global_p95:.6g}, breach_ratio={r.max_breach_ratio:.3f}"
        )
    examples_text = "\n  • ".join(examples) if examples else "none"

    # Rows that share the absolute earliest timestamp (limit to 20) — report their count
    earliest_rows_count = int(earliest_rows.shape[0]) if hasattr(earliest_rows, "shape") else 0

    summary = (
        f"Trace anomaly summary:\n"
        f"- Total trace groups scanned with anomalies in-window: {total_anomaly_groups}.\n"
        f"- Earliest anomaly time (epoch): {earliest_overall}  =>  {earliest_local_str} (Asia/Shanghai, UTC+8).\n"
        f"- Number of groups with that earliest timestamp: {earliest_rows_count} (listing top examples below).\n\n"
        f"Top examples (sorted by earliest anomaly then breach ratio):\n  • {examples_text}\n\n"
        f"Notes:\n"
        f"- The returned table was ordered primarily by earliest_anomaly_timestamp ascending and secondarily by max_breach_ratio descending.\n"
        f"- Many groups show anomalies at the very start of the window (epoch {earliest_overall}).\n"
        f"- Request the 'earliest_rows' or 'anomalies_ordered' DataFrames for the full top-20 detail."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  global_p95  max_breach_ratio
195  dockerA1           trace.self.duration_mean               30              2                  1616504400             0.022012    0.014862          0.481103
66       MG01    trace.to_dockerA1.duration_mean               30              2                  1616504400             0.194897    0.131724          0.479584
189  dockerA1      trace.from_MG01.duration_mean               30              2                  1616504400             0.194897    0.131724          0.479584
117      MG02    trace.to_dockerB2.duration_mean               30              2                  1616504400             0.585472    0.396675          0.475948
240  dockerB2      trace.from_MG02.duration_mean               30              2                  1616504400             0.585472    0.396675          0.475948
10       IG01     trace.to_Tomcat02.duration_p95               31              2                  1616504400             2.053700    1.397300          0.469763
136  Tomcat02       trace.from_IG01.duration_p95               31              2                  1616504400             2.053700    1.397300          0.469763
112      MG02     trace.to_dockerA2.duration_p95               31              2                  1616504400             0.943400    0.711000          0.326864
208  dockerA2       trace.from_MG02.duration_p95               31              2                  1616504400             0.943400    0.711000          0.326864
81       MG02  trace.from_Tomcat02.duration_mean               31              2                  1616504400             0.302500    0.235736          0.283212
150  Tomcat02        trace.to_MG02.duration_mean               31              2                  1616504400             0.302500    0.235736          0.283212
28       IG02     trace.to_Tomcat02.duration_p95               31              2                  1616504400             1.980100    1.573750          0.258205
139  Tomcat02       trace.from_IG02.duration_p95               31              2                  1616504400             1.980100    1.573750          0.258205
110      MG02        trace.to_dockerA1.row_count               30              2                  1616504400            72.000000   57.750000          0.246753
194  dockerA1          trace.from_MG02.row_count               30              2                  1616504400            72.000000   57.750000          0.246753
82       MG02   trace.from_Tomcat02.duration_p95               31              2                  1616504400             1.344500    1.138600          0.180836
151  Tomcat02         trace.to_MG02.duration_p95               31              2                  1616504400             1.344500    1.138600          0.180836
70       MG01     trace.to_dockerA2.duration_p95               31              2                  1616504400             0.845350    0.732850          0.153510
205  dockerA2       trace.from_MG01.duration_p95               31              2                  1616504400             0.845350    0.732850          0.153510
35       IG02        trace.to_Tomcat04.row_count               27              2                  1616504400           149.000000  132.200000          0.127080, 1616504400,       cmdb_id                         trace_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  global_p95  max_breach_ratio
195  dockerA1           trace.self.duration_mean               30              2                  1616504400             0.022012    0.014862          0.481103
66       MG01    trace.to_dockerA1.duration_mean               30              2                  1616504400             0.194897    0.131724          0.479584
189  dockerA1      trace.from_MG01.duration_mean               30              2                  1616504400             0.194897    0.131724          0.479584
117      MG02    trace.to_dockerB2.duration_mean               30              2                  1616504400             0.585472    0.396675          0.475948
240  dockerB2      trace.from_MG02.duration_mean               30              2                  1616504400             0.585472    0.396675          0.475948
10       IG01     trace.to_Tomcat02.duration_p95               31              2                  1616504400             2.053700    1.397300          0.469763
136  Tomcat02       trace.from_IG01.duration_p95               31              2                  1616504400             2.053700    1.397300          0.469763
112      MG02     trace.to_dockerA2.duration_p95               31              2                  1616504400             0.943400    0.711000          0.326864
208  dockerA2       trace.from_MG02.duration_p95               31              2                  1616504400             0.943400    0.711000          0.326864
81       MG02  trace.from_Tomcat02.duration_mean               31              2                  1616504400             0.302500    0.235736          0.283212
150  Tomcat02        trace.to_MG02.duration_mean               31              2                  1616504400             0.302500    0.235736          0.283212
28       IG02     trace.to_Tomcat02.duration_p95               31              2                  1616504400             1.980100    1.573750          0.258205
139  Tomcat02       trace.from_IG02.duration_p95               31              2                  1616504400             1.980100    1.573750          0.258205
110      MG02        trace.to_dockerA1.row_count               30              2                  1616504400            72.000000   57.750000          0.246753
194  dockerA1          trace.from_MG02.row_count               30              2                  1616504400            72.000000   57.750000          0.246753
82       MG02   trace.from_Tomcat02.duration_p95               31              2                  1616504400             1.344500    1.138600          0.180836
151  Tomcat02         trace.to_MG02.duration_p95               31              2                  1616504400             1.344500    1.138600          0.180836
70       MG01     trace.to_dockerA2.duration_p95               31              2                  1616504400             0.845350    0.732850          0.153510
205  dockerA2       trace.from_MG01.duration_p95               31              2                  1616504400             0.845350    0.732850          0.153510
35       IG02        trace.to_Tomcat04.row_count               27              2                  1616504400           149.000000  132.200000          0.127080)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series (store in kernel)
try:
    metric_global_p95_df
except NameError:
    metric_global_p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Filter metric.csv to rows with timestamp == 1616504400
ts_target = 1616504400
metric_at_ts = metric_df[metric_df['timestamp'] == ts_target].copy()

# 3) Merge global_p95 into those rows and compute per-cmdb_id aggregates
metric_at_ts = metric_at_ts.merge(metric_global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Compute breach_ratio per row; handle global_p95 == 0 or NaN
metric_at_ts['breach_ratio'] = np.where(
    (metric_at_ts['global_p95'].isna()) | (metric_at_ts['global_p95'] == 0),
    np.nan,
    (metric_at_ts['value'] - metric_at_ts['global_p95']) / metric_at_ts['global_p95']
)

# Per-cmdb aggregates
cmdb_agg = (
    metric_at_ts
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_kpis_at_ts=('kpi_name', 'nunique'),
        num_kpis_value_ge_p95=('value', lambda s: int(((s >= metric_at_ts.loc[s.index, 'global_p95']).fillna(False)).sum())),
        max_breach_ratio=('breach_ratio', 'max')
    )
)

# Sort by max_breach_ratio desc and limit to top 20
cmdb_agg_sorted = cmdb_agg.sort_values(by='max_breach_ratio', ascending=False).head(20)

# 4) For top 5 components by max_breach_ratio, get their top 5 KPI rows at that timestamp
top5_cmdbs = cmdb_agg_sorted.head(5)['cmdb_id'].tolist()

top_kpis_for_top5 = (
    metric_at_ts[metric_at_ts['cmdb_id'].isin(top5_cmdbs)]
    [['cmdb_id', 'kpi_name', 'value', 'global_p95', 'breach_ratio']]
    .sort_values(['cmdb_id', 'breach_ratio'], ascending=[True, False])
)

# Limit to top 5 per component, overall cap 25
top_kpis_for_top5 = top_kpis_for_top5.groupby('cmdb_id').head(5).reset_index(drop=True).head(25)

# 5) Additionally return all metric rows at timestamp for specific components (limit printed rows to 50)
specific_components = ['MG01', 'MG02', 'IG01', 'Tomcat02', 'Tomcat01', 'Tomcat03']
selected_components_rows = metric_at_ts[metric_at_ts['cmdb_id'].isin(specific_components)][
    ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'breach_ratio']
].head(50)

# Final compact outputs (variables for IPython display)
cmdb_agg_sorted, top_kpis_for_top5, selected_components_rows

```
Out[5]:
```
```python
# Build a concise plain-English summary from existing kernel results (cmdb_agg_sorted, top_kpis_for_top5, selected_components_rows, metric_at_ts)
import pandas as pd
import numpy as np

try:
    cmdb_df = cmdb_agg_sorted
    topk_df = top_kpis_for_top5
    sel_comp_df = selected_components_rows
    all_at_ts = metric_at_ts
except NameError:
    summary = "Required intermediate results are not available in the kernel."
else:
    # total components and total KPI rows at the timestamp
    total_components = int(all_at_ts['cmdb_id'].nunique())
    total_kpi_rows = int(all_at_ts.shape[0])

    # Top 3 components by max_breach_ratio from the provided (already-sorted) table
    top3 = []
    for _, r in cmdb_df.head(3).iterrows():
        top3.append(f"{r.cmdb_id} (max_breach_ratio={r.max_breach_ratio:.3f}, kpis_at_ts={int(r.total_kpis_at_ts)}, kpis>=p95={int(r.num_kpis_value_ge_p95)})")
    top3_text = "; ".join(top3) if top3 else "none"

    # For the top 5 components, summarise how many top-KPI rows we have and the single highest KPI breach_ratio among them
    top5_list = topk_df['cmdb_id'].unique().tolist()
    top5_lines = []
    for comp in top5_list:
        sub = topk_df[topk_df['cmdb_id'] == comp]
        if not sub.empty:
            max_br = sub['breach_ratio'].max()
            kcount = int(sub.shape[0])
            top_kpi = sub.sort_values('breach_ratio', ascending=False).iloc[0]
            top5_lines.append(f"{comp}: {kcount} top KPIs shown, highest breach {max_br:.3f} on '{top_kpi.kpi_name}' (value={top_kpi.value:.6g}, p95={top_kpi.global_p95:.6g})")

    top5_text = "\n  • ".join(top5_lines) if top5_lines else "none"

    # For the requested specific components, report how many KPI rows present and the largest breach among them
    specific = ['MG01', 'MG02', 'IG01', 'Tomcat02', 'Tomcat01', 'Tomcat03']
    spec_lines = []
    for comp in specific:
        sub = sel_comp_df[sel_comp_df['cmdb_id'] == comp]
        if sub.shape[0] == 0:
            spec_lines.append(f"{comp}: no rows at timestamp")
        else:
            max_br = sub['breach_ratio'].max()
            nrows = int(sub.shape[0])
            spec_lines.append(f"{comp}: {nrows} rows, max breach_ratio={max_br:.3f}")

    spec_text = "; ".join(spec_lines)

    # Compose final summary
    summary = (
        f"Metric snapshot at timestamp 1616504400 (compact):\n"
        f"- Total components present: {total_components}; total KPI rows: {total_kpi_rows}.\n"
        f"- Top components by max breach ratio (top 3): {top3_text}.\n\n"
        f"- Top-5 components detail (top KPIs shown):\n  • {top5_text}\n\n"
        f"- Selected components at this timestamp: {spec_text}.\n\n"
        f"Notes: 'breach_ratio' = (value - global_p95) / global_p95; NaN indicates missing or zero global_p95.\n"
        f"To inspect exact rows, examine variables: cmdb_agg_sorted (per-component summary), top_kpis_for_top5 (top KPI rows),\n"
        f"and selected_components_rows (all rows for MG01, MG02, IG01, Tomcat02, Tomcat01, Tomcat03 at this timestamp)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  total_kpis_at_ts  num_kpis_value_ge_p95  max_breach_ratio
4         Mysql01               132                     61          1.439521
5         Mysql02               129                     52          1.253182
19       Tomcat01                80                     46          1.000000
0            IG01                13                      8          0.818182
11   ServiceTest2                 4                      3          0.597938
7         Redis02                65                     48          0.162790
10  ServiceTest11                 4                      3          0.059561
3            MG02                32                     18          0.039091
22       Tomcat04                41                     28          0.001601
2            MG01                30                     16          0.000584
8    ServiceTest1                 4                      2          0.000000
1            IG02                 8                      2          0.000000
6         Redis01                 2                      1          0.000000
13   ServiceTest4                 4                      2          0.000000
14   ServiceTest5                 4                      2          0.000000
9   ServiceTest10                 4                      2          0.000000
12   ServiceTest3                 4                      2          0.000000
16   ServiceTest7                 4                      2          0.000000
15   ServiceTest6                 4                      2          0.000000
18   ServiceTest9                 4                      2          0.000000,      cmdb_id                                           kpi_name         value    global_p95  breach_ratio
0       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  1.000000e+00  5.500000e-01      0.818182
1       IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1.900000e+01  1.500000e+01      0.266667
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09      0.000000
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.863888e+07  5.863888e+07      0.000000
4       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01  7.600000e+01      0.000000
..       ...                                                ...           ...           ...           ...
19  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00  1.500000e+00      1.000000
20  Tomcat01                             OSLinux-CPU_CPU_CPUWio  1.089300e+00  6.506605e-01      0.674145
21  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite  1.304000e+02  1.221867e+02      0.067220
22  Tomcat01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  5.930000e+02  5.920000e+02      0.001689
23  Tomcat01  OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPer...  1.636698e+01  1.636698e+01      0.000000

[24 rows x 5 columns],      timestamp cmdb_id                                     kpi_name         value    global_p95  breach_ratio
0   1616504400    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09      0.000000
1   1616504400    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.142284e+08  8.531137e+08     -0.631669
2   1616504400    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.863888e+07  5.863888e+07      0.000000
3   1616504400    IG01              JVM-Runtime_7778_JVM_JVM_Uptime  1.583445e+06  1.585187e+06     -0.001099
4   1616504400    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01  7.600000e+01      0.000000
..         ...     ...                                          ...           ...           ...           ...
53  1616504400    MG02  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.837965e+07  5.837965e+07      0.000000
54  1616504400    MG02              JVM-Runtime_7779_JVM_JVM_Uptime  1.123440e+05  1.140861e+05     -0.015270
55  1616504400    MG02     JVM-Threads_7779_JVM_ThreadCount_Threads  7.400000e+01  7.400000e+01      0.000000
56  1616504400    MG02                      OSLinux-CPU_CPU_CPULoad  1.210000e+00  1.235000e+00     -0.020243
57  1616504400    MG02                   OSLinux-CPU_CPU_CPUSysTime  7.237330e-01  8.253000e-01     -0.123067

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df and metric_global_p95_df if present; otherwise load/compute
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

try:
    metric_global_p95_df
except NameError:
    metric_global_p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# Candidate components
candidates = ['Mysql01', 'Mysql02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'IG01', 'MG01', 'MG02', 'dockerA1', 'dockerB2']

# Incident window in UTC+8 -> convert to UTC epoch seconds
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# 2) Filter metric points to the incident window and candidate components
metric_window_candidates = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global p95 (computed from full series) into windowed rows
metric_window_candidates = metric_window_candidates.merge(
    metric_global_p95_df, on=['cmdb_id', 'kpi_name'], how='left'
)

# Mark anomalies (value >= global_p95) - only if global_p95 is not NaN
metric_window_candidates['is_anomaly'] = (
    (~metric_window_candidates['global_p95'].isna()) &
    (metric_window_candidates['value'] >= metric_window_candidates['global_p95'])
)

# Helper to compute longest consecutive run (in minutes) given sorted timestamps (seconds)
def longest_consecutive_run_seconds(ts_array, step_seconds=60):
    if len(ts_array) == 0:
        return 0
    ts_sorted = np.sort(np.unique(ts_array.astype('int64')))
    if ts_sorted.size == 0:
        return 0
    if ts_sorted.size == 1:
        return 1
    diffs = np.diff(ts_sorted)
    # run id increments when diff != step_seconds
    run_id = np.concatenate([[0], np.cumsum(diffs != step_seconds)])
    run_lengths = np.bincount(run_id)
    return int(run_lengths.max())

# 3) Compute KPI-level stats for each (cmdb_id, kpi_name) within candidates
def compute_kpi_stats(g):
    # g is windowed rows for one (cmdb_id, kpi_name)
    anomaly_ts = g.loc[g['is_anomaly'], 'timestamp'].to_numpy(dtype='int64')
    anomaly_count = int(len(anomaly_ts))
    earliest_anomaly_timestamp = int(anomaly_ts.min()) if anomaly_count > 0 else pd.NA
    longest_run = longest_consecutive_run_seconds(anomaly_ts, step_seconds=60)
    max_value_in_window = g['value'].max() if not g['value'].isna().all() else np.nan
    global_p95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else np.nan
    if pd.isna(global_p95) or global_p95 == 0:
        max_breach_ratio = np.nan
    else:
        max_breach_ratio = (max_value_in_window - global_p95) / global_p95
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'longest_consecutive_anomaly_run': longest_run,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95,
        'max_breach_ratio': max_breach_ratio
    })

kpi_grouped = metric_window_candidates.groupby(['cmdb_id', 'kpi_name'], as_index=False)
kpi_stats_df = kpi_grouped.apply(lambda g: compute_kpi_stats(g)).reset_index()

# Ensure types
kpi_stats_df['anomaly_count'] = kpi_stats_df['anomaly_count'].astype(int)
kpi_stats_df['longest_consecutive_anomaly_run'] = kpi_stats_df['longest_consecutive_anomaly_run'].astype(int)

# Store full intermediate in kernel variable: kpi_stats_df (already stored)

# 4A) Component-level summary for the candidate list
comp_summary = (
    kpi_stats_df
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_kpis_scanned=('kpi_name', 'nunique'),
        kpis_with_long_run_ge_3=('longest_consecutive_anomaly_run', lambda s: int((s >= 3).sum())),
        component_max_breach_ratio=('max_breach_ratio', lambda s: np.nan if s.dropna().empty else float(s.replace([np.inf, -np.inf], np.nan).max())),
        earliest_component_anomaly=('earliest_anomaly_timestamp', lambda s: int(s.dropna().min()) if s.dropna().shape[0] > 0 else pd.NA)
    )
)

# Ensure candidate list order and include candidates with zero KPIs (if any) by reindexing
comp_summary = comp_summary.set_index('cmdb_id').reindex(candidates).reset_index()
# Fill missing total_kpis_scanned with 0 and others with appropriate defaults
comp_summary['total_kpis_scanned'] = comp_summary['total_kpis_scanned'].fillna(0).astype(int)
comp_summary['kpis_with_long_run_ge_3'] = comp_summary['kpis_with_long_run_ge_3'].fillna(0).astype(int)
comp_summary['component_max_breach_ratio'] = comp_summary['component_max_breach_ratio'].astype(float)
# Sort by kpis_with_long_run_ge_3 desc then component_max_breach_ratio desc
comp_summary_sorted = comp_summary.sort_values(by=['kpis_with_long_run_ge_3', 'component_max_breach_ratio'], ascending=[False, False])

# Limit to candidate list (at most 10 rows) - already limited
component_level_result = comp_summary_sorted[['cmdb_id', 'total_kpis_scanned', 'kpis_with_long_run_ge_3', 'component_max_breach_ratio', 'earliest_component_anomaly']].reset_index(drop=True)

# 4B) KPI-level detail: top 20 KPI rows across these components sorted by longest_consecutive_anomaly_run desc then max_breach_ratio desc
kpi_level_result = (
    kpi_stats_df
    .sort_values(by=['longest_consecutive_anomaly_run', 'max_breach_ratio'], ascending=[False, False])
    [['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'longest_consecutive_anomaly_run', 'max_value_in_window', 'global_p95', 'max_breach_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# Store full tables in kernel variables for further inspection:
kpi_stats_full = kpi_stats_df.copy()
component_summary_full = comp_summary.copy()

# Display compact results (variable names for IPython)
component_level_result, kpi_level_result

```
Out[6]:
```
```python
# Build a concise plain-English summary from the latest kernel results for the candidate components.
import pandas as pd

try:
    component_df = component_level_result.copy()
    kpi_df = kpi_level_result.copy()
except NameError:
    summary = "Required results not found in the kernel. Please run the analysis to produce component_level_result and kpi_level_result."
else:
    # Basic counts
    total_candidates = int(component_df.shape[0])
    comps_with_activity = int(component_df['total_kpis_scanned'].gt(0).sum())

    # Top components by KPIs with long runs (>=3)
    top_by_longrun = component_df.sort_values(['kpis_with_long_run_ge_3', 'component_max_breach_ratio'], ascending=[False, False]).head(5)
    top_longrun_lines = [
        f"{row.cmdb_id}: {row.kpis_with_long_run_ge_3} KPIs with >=3min consecutive anomalies, max_breach={row.component_max_breach_ratio:.3f}"
        for _, row in top_by_longrun.iterrows()
    ]

    # Component with highest max_breach_ratio
    comp_max_breach_row = component_df.loc[component_df['component_max_breach_ratio'].idxmax()]
    comp_max_breach_line = f"{comp_max_breach_row.cmdb_id} (max_breach_ratio={comp_max_breach_row.component_max_breach_ratio:.3f})"

    # Earliest anomaly (common across many components)
    earliest_ts_vals = component_df['earliest_component_anomaly'].dropna()
    earliest_overall = int(earliest_ts_vals.min()) if not earliest_ts_vals.empty else None

    # Top KPI examples (from kpi-level detail shown)
    kpi_examples = []
    for _, r in kpi_df.head(5).iterrows():
        kpi_examples.append(
            f"{r.cmdb_id} | {r.kpi_name} — run={int(r.longest_consecutive_anomaly_run)}m, anomalies={int(r.anomaly_count)}, max_breach={r.max_breach_ratio:.3f}"
        )

    # Components with no KPIs in-window
    no_rows = component_df[component_df['total_kpis_scanned'] == 0]['cmdb_id'].tolist()

    summary_lines = [
        f"Metric-level fault identification for {total_candidates} candidate components (window Mar 23 2021 21:00–21:30 UTC+8):",
        f"- Components with any KPI activity in-window: {comps_with_activity}",
        f"- Earliest anomaly timestamp across candidates: {earliest_overall} (epoch seconds) — many anomalies begin at the window start.",
        "",
        "Top components by count of KPIs with long consecutive anomaly runs (>=3 minutes):",
        *["  • " + line for line in top_longrun_lines],
        "",
        f"Component with largest single KPI breach ratio: {comp_max_breach_line}",
        "",
        "Top KPI examples (highest longest runs / breaches):",
        *["  • " + ex for ex in kpi_examples],
        "",
        f"Components with no KPI rows in-window: {', '.join(no_rows) if no_rows else 'none'}",
        "",
        "Interpretation: Mysql01/Mysql02 and Tomcat nodes show many KPIs with multi-minute consecutive anomalies; MG01 shows the largest component-level breach ratio, indicating a strong metric deviation there. DockerA1 and dockerB2 had no metric rows in the window.",
        "For details, inspect kernel tables: component_level_result (per-component summary) and kpi_level_result (top KPI-level rows)."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_kpis_scanned  kpis_with_long_run_ge_3  component_max_breach_ratio  earliest_component_anomaly
0   Mysql01                 137                       84                    0.540020                1.616504e+09
1   Mysql02                 137                       70                    1.250000                1.616504e+09
2  Tomcat02                 122                       23                    1.000000                1.616504e+09
3  Tomcat03                 123                       22                    2.333333                1.616504e+09
4  Tomcat01                 120                       20                    2.333333                1.616504e+09
5      MG01                 104                        9                    4.285944                1.616504e+09
6      MG02                 106                        9                    1.104101                1.616504e+09
7      IG01                  18                        3                    0.818182                1.616504e+09
8  dockerA1                   0                        0                         NaN                         NaN
9  dockerB2                   0                        0                         NaN                         NaN,      cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  longest_consecutive_anomaly_run  max_value_in_window    global_p95  max_breach_ratio
0       MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                1.616504e+09                               31         3.000000e+00  2.000000e+00          0.500000
1       MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31                1.616504e+09                               31         4.000000e-01  3.000000e-01          0.333333
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31                1.616504e+09                               31         2.040136e+09  2.040136e+09          0.000000
3       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31                1.616504e+09                               31         2.040136e+09  2.040136e+09          0.000000
4       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31                1.616504e+09                               31         5.907150e+07  5.907150e+07          0.000000
5       MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                1.616504e+09                               31         3.000000e+00  3.000000e+00          0.000000
6       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31                1.616504e+09                               31         2.040136e+09  2.040136e+09          0.000000
7       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31                1.616504e+09                               31         5.837965e+07  5.837965e+07          0.000000
8       MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                1.616504e+09                               31         1.000000e+00  1.000000e+00          0.000000
9       MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31                1.616504e+09                               31         3.000000e-01  3.000000e-01          0.000000
10      MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                1.616504e+09                               31         3.000000e+00  3.000000e+00          0.000000
11  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31                1.616504e+09                               31         5.600000e+02  5.600000e+02          0.000000
12  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31                1.616504e+09                               31         3.978080e+05  3.978080e+05          0.000000
13  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31                1.616504e+09                               31         1.000000e+02  1.000000e+02          0.000000
14  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31                1.616504e+09                               31         5.000000e+02  5.000000e+02          0.000000
15  Tomcat02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                1.616504e+09                               31         1.000000e+00  1.000000e+00          0.000000
16  Tomcat02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                1.616504e+09                               31         3.000000e+00  3.000000e+00          0.000000
17  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31                1.616504e+09                               31         5.150000e+02  5.150000e+02          0.000000
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31                1.616504e+09                               31         3.906190e+05  3.906190e+05          0.000000
19  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31                1.616504e+09                               31         1.000000e+02  1.000000e+02          0.000000)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure numeric types
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (store in kernel variable)
try:
    trace_global_p95_df
except NameError:
    trace_global_p95_df = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Filter trace.csv to the incident window Mar 23, 2021 21:00–21:30 (inclusive) (UTC+8 -> convert to UTC)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Select groups: cmdb_id == MG01 OR trace_name contains 'from_MG01'/'to_MG01' OR contains endpoints Tomcat01/02/03, IG01, Mysql01, Mysql02
candidates_endpoints = ['Tomcat01', 'Tomcat02', 'Tomcat03', 'IG01', 'Mysql01', 'Mysql02']
mask = (
    (trace_window['cmdb_id'] == 'MG01') |
    trace_window['trace_name'].str.contains('from_MG01', na=False) |
    trace_window['trace_name'].str.contains('to_MG01', na=False)
)
# include any trace_name containing candidate endpoint substrings
for ep in candidates_endpoints:
    mask = mask | trace_window['trace_name'].str.contains(ep, na=False)

trace_selected = trace_window[mask].copy()

# Merge global p95 thresholds (computed from full series) into selected windowed rows
trace_selected = trace_selected.merge(trace_global_p95_df, on=['cmdb_id', 'trace_name'], how='left')

# Mark anomalies where value >= global_p95 (only when global_p95 is not NaN)
trace_selected['is_anomaly'] = (~trace_selected['global_p95'].isna()) & (trace_selected['value'] >= trace_selected['global_p95'])

# Helper: longest consecutive run in minutes given array of epoch-second timestamps (expected per-minute series)
def longest_consecutive_run_minutes(ts_array, step_seconds=60):
    if len(ts_array) == 0:
        return 0
    ts_sorted = np.sort(np.unique(ts_array.astype('int64')))
    if ts_sorted.size == 0:
        return 0
    if ts_sorted.size == 1:
        return 1
    diffs = np.diff(ts_sorted)
    run_id = np.concatenate([[0], np.cumsum(diffs != step_seconds)])
    run_lengths = np.bincount(run_id)
    return int(run_lengths.max())

# 4) Aggregate per (cmdb_id, trace_name)
grouped = trace_selected.groupby(['cmdb_id', 'trace_name'], as_index=False)

def compute_trace_stats(g):
    count_in_window = int(g['value'].count())
    anomaly_ts = g.loc[g['is_anomaly'], 'timestamp'].to_numpy(dtype='int64')
    anomaly_count = int(len(anomaly_ts))
    earliest_anomaly_timestamp = int(anomaly_ts.min()) if anomaly_count > 0 else pd.NA
    longest_run = longest_consecutive_run_minutes(anomaly_ts, step_seconds=60)
    max_value_in_window = g['value'].max() if not g['value'].isna().all() else np.nan
    global_p95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else np.nan
    if pd.isna(global_p95) or global_p95 == 0:
        max_breach_ratio = np.nan
    else:
        max_breach_ratio = (max_value_in_window - global_p95) / global_p95
    return pd.Series({
        'count_in_window': count_in_window,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'longest_consecutive_anomaly_run': longest_run,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95,
        'max_breach_ratio': max_breach_ratio
    })

trace_stats = grouped.apply(lambda g: compute_trace_stats(g)).reset_index()

# Store full intermediate in kernel variables
trace_selected_full = trace_selected.copy()
trace_stats_full = trace_stats.copy()

# 5) Prepare compact table sorted by earliest_anomaly_timestamp asc then max_breach_ratio desc; select top 20
# When earliest_anomaly_timestamp is NA, sort them last
trace_stats_sorted = trace_stats.sort_values(
    by=['earliest_anomaly_timestamp', 'max_breach_ratio'],
    ascending=[True, False],
    na_position='last'
)

selected_stats_top20 = trace_stats_sorted[[
    'cmdb_id', 'trace_name', 'count_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'longest_consecutive_anomaly_run',
    'max_value_in_window', 'global_p95', 'max_breach_ratio'
]].head(20).reset_index(drop=True)

# 6) One-line summary: number of trace groups scanned and number with anomaly_count > 0 within this selected set
total_trace_groups_scanned = int(trace_stats.shape[0])
trace_groups_with_anomalies = int((trace_stats['anomaly_count'] > 0).sum())
summary = (f"Scanned {total_trace_groups_scanned} selected trace groups; "
           f"{trace_groups_with_anomalies} groups had anomaly_count>0 in the window {start_local} to {end_local} (UTC+8).")

# Return compact table (selected_stats_top20) and summary
selected_stats_top20, summary

```
Out[7]:
```
```python
# Build a concise plain-English summary from the last computed results in the kernel
try:
    df = selected_stats_top20.copy()
    total_scan_info = summary  # one-line summary from earlier
except NameError:
    df = None
    total_scan_info = None

if df is None or total_scan_info is None:
    result = "Trace results not found in kernel; please run the selection/aggregation step first."
else:
    # overall counts from the previously produced summary string (fallback to df counts)
    try:
        # extract numbers from the existing summary string for accuracy
        scanned_part = total_scan_info.split(";")[0]
    except Exception:
        scanned_part = ""
    # top 3 groups by breach ratio among the displayed top-20
    top_breaches = df.sort_values('max_breach_ratio', ascending=False).head(3)
    top_lines = []
    for _, r in top_breaches.iterrows():
        top_lines.append(
            f"{r.cmdb_id} | {r.trace_name} — anomaly_count={int(r.anomaly_count)}, "
            f"longest_run={int(r.longest_consecutive_anomaly_run)}m, breach_ratio={r.max_breach_ratio:.3f}"
        )
    # distribution summary
    avg_run = float(df['longest_consecutive_anomaly_run'].mean()) if not df.empty else 0.0
    median_anom = float(df['anomaly_count'].median()) if not df.empty else 0.0
    earliest_ts = int(df['earliest_anomaly_timestamp'].min()) if not df.empty else None

    result = (
        f"{scanned_part}\n\n"
        "Summary (plain English):\n"
        f"- Selected trace groups scanned: {int(df.shape[0])} shown (subset top-20); refer to full selection for all groups.\n"
        f"- All selected groups had anomalies in-window (anomaly_count>0). Most groups show anomaly_count of 2 (median={median_anom:.0f}).\n"
        f"- Longest consecutive anomaly runs are short on average (mean ≈ {avg_run:.2f} minutes); many runs are 1 minute long.\n"
        f"- Earliest anomalies generally start at epoch {earliest_ts} (this is the window start).\n\n"
        "Top breach examples (highest relative exceedance among displayed):\n  • " + ("\n  • ".join(top_lines)) + "\n\n"
        "Interpretation:\n"
        "- The selection around MG01 shows multiple traces between MG01 and other services (Tomcat, dockerA/B, IG01) with brief anomaly occurrences during the incident window.\n"
        "- Relative breach ratios up to ~0.83 were observed (i.e., ~83% above historical P95) for some trace P95 metrics, and several MG01-related traces exceed P95 by ~48%.\n"
        "- Most anomalies are short (1–2 minutes) rather than long sustained runs, though some KPIs show longer consecutive runs and merit deeper inspection.\n\n"
        "If you want, I can return the full trace_stats table (trace_stats_full) or the full selected trace points (trace_selected_full) for deeper investigation."
    )

result
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  count_in_window  anomaly_count  earliest_anomaly_timestamp  longest_consecutive_anomaly_run  max_value_in_window  global_p95  max_breach_ratio
0       MG01    trace.to_dockerA1.duration_mean             30.0            2.0                1.616504e+09                              1.0             0.194897    0.131724          0.479584
1   dockerA1      trace.from_MG01.duration_mean             30.0            2.0                1.616504e+09                              1.0             0.194897    0.131724          0.479584
2       IG01     trace.to_Tomcat02.duration_p95             31.0            2.0                1.616504e+09                              1.0             2.053700    1.397300          0.469763
3   Tomcat02       trace.from_IG01.duration_p95             31.0            2.0                1.616504e+09                              1.0             2.053700    1.397300          0.469763
4       MG02  trace.from_Tomcat02.duration_mean             31.0            2.0                1.616504e+09                              1.0             0.302500    0.235736          0.283212
5       IG02     trace.to_Tomcat02.duration_p95             31.0            2.0                1.616504e+09                              1.0             1.980100    1.573750          0.258205
6       MG02   trace.from_Tomcat02.duration_p95             31.0            2.0                1.616504e+09                              1.0             1.344500    1.138600          0.180836
7       MG01     trace.to_dockerA2.duration_p95             31.0            2.0                1.616504e+09                              1.0             0.845350    0.732850          0.153510
8   dockerA2       trace.from_MG01.duration_p95             31.0            2.0                1.616504e+09                              1.0             0.845350    0.732850          0.153510
9       MG01     trace.to_dockerA1.duration_p95             30.0            2.0                1.616504e+09                              1.0             1.033300    0.970307          0.064920
10  dockerA1       trace.from_MG01.duration_p95             30.0            2.0                1.616504e+09                              1.0             1.033300    0.970307          0.064920
11      MG01  trace.from_Tomcat04.duration_mean             27.0            2.0                1.616504e+09                              1.0             0.229030    0.215665          0.061973
12  Tomcat04        trace.to_MG01.duration_mean             27.0            2.0                1.616504e+09                              1.0             0.229030    0.215665          0.061973
13      IG01    trace.to_Tomcat02.duration_mean             31.0            2.0                1.616504e+09                              1.0             0.355358    0.340733          0.042922
14  Tomcat02      trace.from_IG01.duration_mean             31.0            2.0                1.616504e+09                              1.0             0.355358    0.340733          0.042922
15      MG01      trace.from_Tomcat04.row_count             27.0            2.0                1.616504e+09                              1.0           136.000000  131.700000          0.032650
16  Tomcat04            trace.to_MG01.row_count             27.0            2.0                1.616504e+09                              1.0           136.000000  131.700000          0.032650
17      MG01        trace.to_dockerA2.row_count             31.0            3.0                1.616504e+09                              2.0           179.000000  178.000000          0.005618
18  dockerA2          trace.from_MG01.row_count             31.0            3.0                1.616504e+09                              2.0           179.000000  178.000000          0.005618
19      MG02   trace.from_Tomcat01.duration_p95             31.0            2.0                1.616504e+09                              1.0             1.986000    1.082375          0.834854, 'Scanned 117 selected trace groups; 117 groups had anomaly_count>0 in the window 2021-03-23 21:00:00+08:00 to 2021-03-23 21:30:00+08:00 (UTC+8).')```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric types
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

# 1) Ensure global P95 per (cmdb_id, kpi_name) is computed from full series (store in kernel)
try:
    metric_global_p95_df
except NameError:
    metric_global_p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_p95'})
    )

# 2) Filter metric.csv to MG01 rows inside the incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(pd.Timestamp("2021-03-23 21:00:00"))
end_local = tz.localize(pd.Timestamp("2021-03-23 21:30:00"))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

mg01_metric_window = metric_df[
    (metric_df['cmdb_id'] == 'MG01') &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Merge global p95 thresholds computed from full series
mg01_metric_window = mg01_metric_window.merge(metric_global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global_p95 (only when global_p95 is not NaN)
mg01_metric_window['is_anomaly'] = (~mg01_metric_window['global_p95'].isna()) & (mg01_metric_window['value'] >= mg01_metric_window['global_p95'])

# Helper: longest consecutive run (in minutes) given anomaly timestamps (expected per-minute series)
def longest_consecutive_run_minutes(ts_array, step_seconds=60):
    if len(ts_array) == 0:
        return 0
    ts_sorted = np.sort(np.unique(ts_array.astype('int64')))
    if ts_sorted.size == 0:
        return 0
    if ts_sorted.size == 1:
        return 1
    diffs = np.diff(ts_sorted)
    run_id = np.concatenate([[0], np.cumsum(diffs != step_seconds)])
    run_lengths = np.bincount(run_id)
    return int(run_lengths.max())

# 3) For each MG01 kpi_name compute the requested KPI-level stats
def compute_kpi_stats(g):
    anomaly_ts = g.loc[g['is_anomaly'], 'timestamp'].to_numpy(dtype='int64')
    anomaly_count = int(anomaly_ts.size)
    earliest_anomaly_timestamp = int(anomaly_ts.min()) if anomaly_count > 0 else pd.NA
    longest_run = longest_consecutive_run_minutes(anomaly_ts, step_seconds=60)
    max_value_in_window = g['value'].max() if not g['value'].isna().all() else np.nan
    global_p95 = g['global_p95'].iloc[0] if 'global_p95' in g.columns else np.nan
    if pd.isna(global_p95) or global_p95 == 0:
        max_breach_ratio = np.nan
    else:
        max_breach_ratio = (max_value_in_window - global_p95) / global_p95
    return pd.Series({
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'longest_consecutive_anomaly_run': longest_run,
        'max_value_in_window': max_value_in_window,
        'global_p95': global_p95,
        'max_breach_ratio': max_breach_ratio
    })

mg01_kpi_stats_df = (
    mg01_metric_window
    .groupby('kpi_name', as_index=False)
    .apply(lambda g: compute_kpi_stats(g))
    .reset_index()
)

# Ensure integer types for counts/runs
mg01_kpi_stats_df['anomaly_count'] = mg01_kpi_stats_df['anomaly_count'].astype(int)
mg01_kpi_stats_df['longest_consecutive_anomaly_run'] = mg01_kpi_stats_df['longest_consecutive_anomaly_run'].astype(int)

# 4) Return top 20 MG01 KPI rows sorted by max_breach_ratio desc
mg01_kpi_result = mg01_kpi_stats_df.sort_values(by='max_breach_ratio', ascending=False)[[
    'kpi_name', 'anomaly_count', 'earliest_anomaly_timestamp', 'longest_consecutive_anomaly_run',
    'max_value_in_window', 'global_p95', 'max_breach_ratio'
]].head(20).reset_index(drop=True)

# 5) One-line summary
total_mg01_kpis_scanned = int(mg01_kpi_stats_df.shape[0])
kpis_with_long_run_ge_3 = int((mg01_kpi_stats_df['longest_consecutive_anomaly_run'] >= 3).sum())
summary = f"Scanned {total_mg01_kpis_scanned} MG01 KPIs; {kpis_with_long_run_ge_3} KPIs have longest_consecutive_anomaly_run >= 3 minutes (window {start_local} to {end_local} UTC+8)."

# Store full intermediate tables in kernel variables for follow-ups:
mg01_metric_window_full = mg01_metric_window.copy()
mg01_kpi_stats_full = mg01_kpi_stats_df.copy()

# Display compact outputs (DataFrame and summary string)
mg01_kpi_result, summary

```
Out[8]:
```
```python
# Build a concise plain-English summary from existing kernel results (mg01_kpi_result, summary)
try:
    df = mg01_kpi_result.copy()
    one_line = summary
except NameError:
    df = None
    one_line = None

if df is None or one_line is None:
    result = "MG01 KPI results not found in kernel. Please run the MG01 analysis first."
else:
    top3 = []
    for _, r in df.head(3).iterrows():
        top3.append(
            f"{r.kpi_name} — breach_ratio={r.max_breach_ratio:.3f}, anomalies={int(r.anomaly_count)}, "
            f"longest_run={int(r.longest_consecutive_anomaly_run)}m, max_value={r.max_value_in_window:.6g}"
        )
    top3_text = "\n  • ".join(top3) if top3 else "none"

    result = (
        f"{one_line}\n\n"
        "Key MG01 findings (compact):\n"
        f"- Top KPI breaches (by breach_ratio):\n  • {top3_text}\n\n"
        "- Interpretation: Several MG01 OS and process metrics exceeded their historical P95 during the incident window. "
        "The largest relative breach is OSLinux-CPU_CPU_CPUWio (~4.29x P95). Nine MG01 KPIs had sustained anomalies >= 3 minutes, "
        "indicating persistent metric deviations worth deeper investigation."
    )

result
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  anomaly_count  earliest_anomaly_timestamp  longest_consecutive_anomaly_run  max_value_in_window     global_p95  max_breach_ratio
0                              OSLinux-CPU_CPU_CPUWio              2                1.616504e+09                                2             3.230769       0.611200          4.285944
1      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1                1.616505e+09                                1            11.000000       5.400000          1.037037
2      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps              1                1.616504e+09                                1             1.000000       0.550000          0.818182
3      OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps              1                1.616504e+09                                1             1.000000       0.550000          0.818182
4   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...              1                1.616504e+09                                1           621.000000     354.150000          0.753494
5   OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...              1                1.616505e+09                                1             9.000000       5.850000          0.538462
6       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                1.616504e+09                               31             3.000000       2.000000          0.500000
7                     OSLinux-CPU_CPU-1_SingleCpuUtil              1                1.616505e+09                                1            14.399700      10.138335          0.420322
8   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31                1.616504e+09                               31             0.400000       0.300000          0.333333
9     OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec              1                1.616506e+09                                1        569466.000000  458625.200000          0.241681
10   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite              1                1.616504e+09                                1           621.066700     525.306677          0.182294
11  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              1                1.616504e+09                                1            15.000000      13.200000          0.136364
12                    OSLinux-CPU_CPU-3_SingleCpuidle              1                1.616505e+09                                1            15.691489      13.966337          0.123522
13                         OSLinux-CPU_CPU_CPUCpuUtil              1                1.616505e+09                                1            30.224500      27.900020          0.083315
14                            OSLinux-CPU_CPU_CPULoad              2                1.616506e+09                                1             1.450000       1.345000          0.078067
15   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite              1                1.616505e+09                                1            18.133333      16.920000          0.071710
16    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum              2                1.616506e+09                                1          2575.000000    2476.000000          0.039984
17           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem              1                1.616505e+09                                1           145.000000     140.200000          0.034237
18          JVM-Operating System_7779_JVM_JVM_CPULoad              1                1.616506e+09                                1             1.564400       1.517975          0.030584
19                         OSLinux-CPU_CPU_CPUSysTime              2                1.616506e+09                                1             0.840000       0.817318          0.027752, 'Scanned 104 MG01 KPIs; 9 KPIs have longest_consecutive_anomaly_run >= 3 minutes (window 2021-03-23 21:00:00+08:00 to 2021-03-23 21:30:00+08:00 UTC+8).')```
```